# ⚡ Electricity Bill Prediction — Regression Project
**Goal:** Predict a household's monthly electricity bill from its appliances and household details.

**Project flow:** Load → Understand → Clean → Describe → Explore (EDA) → Hypothesis tests & Correlation → Preprocess → Train 5 models → Evaluate → Save → Streamlit app

> ⚠️ **Important note about the dataset:** the uploaded CSV (`Electricity_Bill_Prediction-selected-columns.csv`) has 1000 rows but **no bill / target column**. So in Step 4 we create a *simulated* `Electricity_Bill` from the appliance columns (+ random noise).
> If you get the original dataset that contains the real bill column, skip that cell and just use the real column — everything after works unchanged.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")

## 2. Load the data
Run this cell and upload `Electricity_Bill_Prediction-selected-columns.csv` when asked.

In [ ]:
from google.colab import files
uploaded = files.upload()          # choose the CSV file

df = pd.read_csv("Electricity_Bill_Prediction-selected-columns.csv")
df.head()

## 3. Understand the data

In [ ]:
print("Shape:", df.shape)
print()
df.info()

In [ ]:
df.describe()

## 4. Data cleaning

In [ ]:
print("Missing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

df = df.drop_duplicates()
df = df.drop(columns=["Customer_ID"])   # ID is not a useful feature
print("\nShape after cleaning:", df.shape)

### Create the target column `Electricity_Bill` (simulated)
The CSV has no bill column, so we build one using a realistic formula (in BDT): AC and refrigerator use the most power, then fans, TVs and lights, plus household size, rooms and area type, plus random noise.
**Skip this cell if your dataset already has a real bill column.**

In [ ]:
np.random.seed(42)

area_extra = df["Area_Type"].map({"Urban": 300, "Semi-Urban": 100, "Rural": 0})

df["Electricity_Bill"] = (
    500
    + 700 * df["AC_Units"]
    + 60  * df["Fan_Count"]
    + 25  * df["Light_Count"]
    + 350 * df["Refrigerator"]
    + 120 * df["TV_Count"]
    + 150 * df["Household_Size"]
    + 100 * df["Rooms"]
    + area_extra
    + np.random.normal(0, 200, len(df))      # random noise
).round(2)

df.head()

## 5. Descriptive statistics (mean, median, mode, variance, std, IQR, 95% CI)

In [ ]:
bill = df["Electricity_Bill"]

print("Mean     :", round(bill.mean(), 2))
print("Median   :", round(bill.median(), 2))
print("Mode     :", bill.mode()[0])
print("Range    :", round(bill.max() - bill.min(), 2))
print("Variance :", round(bill.var(), 2))
print("Std Dev  :", round(bill.std(), 2))

Q1, Q3 = bill.quantile(0.25), bill.quantile(0.75)
print("IQR      :", round(Q3 - Q1, 2))

ci = stats.t.interval(0.95, df=len(bill)-1, loc=bill.mean(), scale=stats.sem(bill))
print("95% Confidence Interval for mean bill:", (round(float(ci[0]), 2), round(float(ci[1]), 2)))

## 6. Exploratory Data Analysis (EDA)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(bill, bins=20, color="skyblue", edgecolor="black")
ax[0].set_title("Histogram of Electricity Bill")
ax[0].set_xlabel("Bill (BDT)")
ax[1].boxplot(bill)
ax[1].set_title("Boxplot of Electricity Bill")
plt.show()

In [ ]:
# Bar chart: average bill by area type
df.groupby("Area_Type")["Electricity_Bill"].mean().plot(kind="bar", color=["orange", "green", "blue"])
plt.title("Average Bill by Area Type")
plt.ylabel("Average Bill (BDT)")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Scatter plot: AC units vs bill
plt.scatter(df["AC_Units"], df["Electricity_Bill"], alpha=0.5)
plt.xlabel("AC Units")
plt.ylabel("Electricity Bill")
plt.title("AC Units vs Electricity Bill")
plt.show()

In [ ]:
# Correlation heatmap (numeric columns only)
plt.figure(figsize=(9, 6))
sns.heatmap(df.select_dtypes(include=np.number).corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

## 7. Correlation and hypothesis testing (α = 0.05)

In [ ]:
# Correlation: AC_Units vs Bill
x, y = df["AC_Units"], df["Electricity_Bill"]
print("Covariance:", round(x.cov(y), 2))
r, p = stats.pearsonr(x, y)
print("Pearson  r =", round(r, 3), " p =", p)
rho, p2 = stats.spearmanr(x, y)
print("Spearman ρ =", round(rho, 3), " p =", p2)

In [ ]:
alpha = 0.05

# 1) One-sample t-test: is the mean bill different from 4000 ?
t, p = stats.ttest_1samp(bill, popmean=4000)
print("One-sample t-test  p =", p, "->", "Reject H0" if p < alpha else "Fail to reject H0")

# 2) Independent t-test: Urban vs Rural bills
urban = df[df["Area_Type"] == "Urban"]["Electricity_Bill"]
rural = df[df["Area_Type"] == "Rural"]["Electricity_Bill"]
t, p = stats.ttest_ind(urban, rural)
print("Independent t-test (Urban vs Rural)  p =", p, "->", "Reject H0" if p < alpha else "Fail to reject H0")

# 3) One-way ANOVA: Urban vs Semi-Urban vs Rural
semi = df[df["Area_Type"] == "Semi-Urban"]["Electricity_Bill"]
f, p = stats.f_oneway(urban, semi, rural)
print("ANOVA (3 area types)  p =", p, "->", "Reject H0" if p < alpha else "Fail to reject H0")

# 4) Chi-square: Area_Type vs 'has AC' (categorical vs categorical)
table = pd.crosstab(df["Area_Type"], df["AC_Units"] > 0)
chi2, p, dof, expected = stats.chi2_contingency(table)
print("Chi-square (Area_Type vs Has AC)  p =", p, "->", "Reject H0" if p < alpha else "Fail to reject H0")

## 8. Preprocessing: encoding, split and scaling

In [ ]:
# One-hot encode categorical columns
df_model = pd.get_dummies(df, columns=["Location", "Area_Type"], drop_first=True)

# X = inputs, y = target (do NOT keep the bill inside X)
X = df_model.drop(columns=["Electricity_Bill"])
y = df_model["Electricity_Bill"]

# Split first, then scale
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled  = scaler.transform(X_test)        # only transform test

print("Train:", X_train.shape, " Test:", X_test.shape)

## 9. Train and compare 5 regression models

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge":             Ridge(alpha=1.0),
    "Lasso":             Lasso(alpha=0.1),
    "Decision Tree":     DecisionTreeRegressor(max_depth=5, random_state=42),
    "Random Forest":     RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
}

results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    mse = mean_squared_error(y_test, pred)
    results.append({
        "Model": name,
        "MAE":  mean_absolute_error(y_test, pred),
        "MSE":  mse,
        "RMSE": np.sqrt(mse),
        "R2":   r2_score(y_test, pred),
    })

results_df = pd.DataFrame(results).sort_values("R2", ascending=False).round(3)
results_df

In [ ]:
results_df.plot(x="Model", y="R2", kind="bar", legend=False, color="teal")
plt.title("R² Score of Each Model (higher is better)")
plt.ylabel("R²")
plt.xticks(rotation=30)
plt.show()

## 10. Best model — actual vs predicted

In [ ]:
best_name  = results_df.iloc[0]["Model"]
best_model = models[best_name]
print("Best model:", best_name)

best_pred = best_model.predict(X_test_scaled)

plt.scatter(y_test, best_pred, alpha=0.5)
plt.plot([y.min(), y.max()], [y.min(), y.max()], "r--")   # perfect-prediction line
plt.xlabel("Actual Bill")
plt.ylabel("Predicted Bill")
plt.title(f"Actual vs Predicted ({best_name})")
plt.show()

In [ ]:
# Which features matter most? (works for Linear/Ridge/Lasso coefficients)
if hasattr(best_model, "coef_"):
    imp = pd.Series(best_model.coef_, index=X.columns).sort_values()
else:
    imp = pd.Series(best_model.feature_importances_, index=X.columns).sort_values()

imp.plot(kind="barh", figsize=(7, 6), color="purple")
plt.title("Feature Importance / Coefficients")
plt.show()

## 11. Save the model, scaler and column names

In [ ]:
joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(list(X.columns), "columns.pkl")

files.download("best_model.pkl")
files.download("scaler.pkl")
files.download("columns.pkl")

## 12. Conclusion
- The dataset had 1000 households with no missing values or duplicates.
- Number of **AC units** and **household size / rooms** have the biggest effect on the bill.
- Urban households pay more on average than rural ones (confirmed by t-test and ANOVA).
- Several models were compared using MAE, MSE, RMSE and R²; the best one is saved and used in the Streamlit app.
- ⚠️ The target bill was simulated, so the scores mainly show the workflow works. With the real bill column, results will differ.

## 13. Run the Streamlit app in Colab
Upload `app.py` to Colab (left Files panel) together with the 3 `.pkl` files, then run these cells:

In [ ]:
!pip install streamlit -q
!npm install localtunnel -q

In [ ]:
# Start streamlit in background, then open a public link
!streamlit run app.py &>/content/logs.txt &

import urllib
print("Password (your Colab IP):", urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode().strip())
!npx localtunnel --port 8501